# HospitalityHub learning notebook: Level 1 and Level 2

This notebook is a learning companion for the HospitalityHub practice project. It records what is built and explains the Angular ideas behind it in beginner-friendly language. Read the Markdown cells from top to bottom; code examples are illustrative and are not executed by this notebook.

**Current status:** Level 1 is substantially complete. Level 2 has started with a global HTTP loading and error interceptor.

Paths in this notebook are relative to the HospitalityHub repository root unless noted otherwise.

## 1. What the application does

HospitalityHub is a small hotel-management app. The Angular client talks to a .NET Web API. The API stores the data in a database through Entity Framework Core.

- **Rooms:** view rooms, add a room, and remove a room. A room has a number, type, nightly price, and availability.
- **Bookings:** select an available room, enter guest details and dates, see an estimated price, save the reservation, and view reservations in the bookings list.
- **Two backend choices:** a SQL Server API and a SQLite API provide the same booking/room endpoints. Run one API at a time and point the Angular `ApiService` at it.
- **Presentation:** responsive room and booking pages share the Poppins / Inter font stack, status messages, and loading/error states.

A saved booking is not just browser state: the client sends it to `POST /api/bookings`, and the .NET API saves it with EF Core. `GET /api/bookings` reads the persisted reservations back.

## 2. Level 1 — CRUD foundations (substantially complete)

CRUD means **Create, Read, Update, Delete**. It describes the basic operations an app performs on stored data.

| Area | Implemented so far |
| --- | --- |
| Rooms | Create a room, read/list rooms, delete a room |
| Bookings | Create a booking, read/list bookings |
| Validation | Required room, guest name, valid email, check-in not in the past, and check-out after check-in |
| Booking rules | The API verifies that the room exists and is available, rejects overlapping stays, and calculates the total from the server's room price |
| Feedback | The UI shows loading, success, validation, and API failure messages |

**Be precise about the word CRUD:** room update is not implemented in the client UI, and booking update/cancellation (delete) are not implemented. The completed operations are useful, but full CRUD for both entities remains a possible follow-up.

### Angular ideas used in Level 1

- **Standalone components** are self-contained UI building blocks; this Angular app does not use feature NgModules.
- **Routes** map browser URLs to screens: `/` for rooms, `/book` for the reservation form, and `/bookings` for the booking register.
- **Dependency injection** lets a component receive the shared `ApiService` rather than constructing HTTP code itself.
- **`HttpClient` and Observables:** calling the API returns an Observable. The app subscribes to the result to update UI state when a response succeeds or fails.
- **Reactive forms:** the form model stores values and validation state separately from the HTML. Validators keep bad or incomplete data from being submitted.
- **Signals:** signals hold UI state such as rooms, loading, and errors. A template reading a signal updates when that signal changes.
- **Backend validation is still required:** browser validation improves usability, but the API must enforce important booking rules because clients can be bypassed.

### Where to explore the Level 1 code

- `hospitality-client/src/app/app.routes.ts` — screen routing.
- `hospitality-client/src/app/core/services/api.ts` — the HTTP service used by features.
- `hospitality-client/src/app/core/models/room.ts` and `booking.ts` — TypeScript data shapes.
- `hospitality-client/src/app/features/rooms/room-list/` — room management UI.
- `hospitality-client/src/app/features/bookings/booking-form/` — validated create-booking flow.
- `hospitality-client/src/app/features/bookings/booking-list/` — persisted booking register.
- `HospitalityHub.Api/Controllers/BookingsController.cs` and `HospitalityHub.Api.Sqlite/Controllers/BookingsController.cs` — booking endpoints for each database provider.

## 3. Level 2 — What is an HTTP interceptor?

Imagine every outgoing parcel passes through a reception desk. The desk can stamp it, record it, or notice that delivery failed. An HTTP interceptor is that reception desk for Angular HTTP requests and responses.

Without an interceptor, every feature repeats the same loading and error plumbing. With an interceptor, one shared function can observe requests made by `HttpClient` and apply cross-cutting behavior consistently.

The simplified flow is:

1. A component calls `ApiService.getRooms()`.
2. `ApiService` calls `HttpClient`.
3. Angular runs the configured interceptor before sending the HTTP request.
4. The response (or HTTP error) travels back through the interceptor.
5. The subscriber in the feature receives the response or the same error.

Interceptors only see requests made through Angular `HttpClient`; they do not intercept browser navigation or arbitrary `fetch()` calls.

### Modern Angular: functional interceptors

This project uses `HttpInterceptorFn`, Angular's functional interceptor type. `withInterceptors([...])` registers it with `provideHttpClient(...)` in `app.config.ts`. Functional interceptors can use Angular dependency injection via `inject()` while the interceptor is running.

Official Angular guide: [HTTP interceptors](https://v21.angular.dev/guide/http/interceptors).

## 4. Loading indicator: why we count requests

Two API calls may happen at the same time (for example, the room list and another screen's data). A boolean is not enough: if request A finishes first, setting `loading = false` would hide the indicator while request B is still running.

Instead, `GlobalLoadingService` holds an **active request count**:

1. When an HTTP request begins, increment the count.
2. When it completes, errors, or is cancelled, decrement it in RxJS `finalize()`.
3. The global indicator is visible exactly while the count is greater than zero.

`finalize()` is important because it runs for success, error, and unsubscribe/cancellation. The service clamps the count at zero as a defensive guard.

The app uses a slim progress line and a small spinner/status pill. The status region and visually hidden text allow assistive technology to announce that data is loading. Because the indicator is global, feature components can still keep their more specific loading or empty states.

## 5. Global error feedback without swallowing errors

The interceptor detects `HttpErrorResponse` and adds a readable message to `GlobalApiErrorService`. The root `GlobalHttpStatusComponent` displays dismissible notifications, so errors from any `HttpClient` call can be seen even when the current page has no dedicated error panel.

**The interceptor rethrows the original error.** `throwError(() => error)` passes it back to the caller instead of pretending the request succeeded. The booking and room screens can therefore still show context-specific inline errors. A global toast and an inline error may both appear for the same failed operation: the toast is the app-wide alert; the inline message explains what failed in that particular form or list.

Typical cases include:

- `status === 0`: the browser could not connect (API offline, certificate, or CORS issue).
- A server response such as 400, 404, or 500: show a safe API problem title/detail when present, otherwise show the HTTP status.

The global notification service keeps a small recent queue and supports a keyboard-accessible dismiss button. Notifications use alert/live-region semantics. Do not put credentials or sensitive request payloads into global error messages or logs.

### Optional next refinement: opt out per request

Some calls may intentionally be quiet, such as background refreshes. Angular supports `HttpContextToken` so an individual request can pass metadata that the interceptor reads. This project does not currently opt out any requests; add that only when a real quiet-request requirement exists.

## 6. Walk through the implementation

### Registration — `hospitality-client/src/app/app.config.ts`

The app calls `provideHttpClient(withInterceptors([loadingErrorInterceptor]))`. If this registration is missing, Angular never runs the interceptor. `provideHttpClient()` remains the provider that makes `HttpClient` available.

### Request/response middleware — `hospitality-client/src/app/core/interceptors/loading-error.interceptor.ts`

The interceptor injects the two shared services, marks the request as active, and returns the Observable from `next(request)`. `catchError` publishes an `HttpErrorResponse` message and rethrows the same error. `finalize` always tells the loading service that this request is finished.

The request object is immutable. If an interceptor needs to change headers or URL, it must create a copy using `request.clone(...)`; this loading/error interceptor does not need to change the request.

### Shared state — `hospitality-client/src/app/core/services/global-loading.ts`

`activeRequestCount` is the source of truth. `isLoading` is a computed signal derived from the count. The interceptor changes the count; the UI reads the computed signal.

### Error queue — `hospitality-client/src/app/core/services/global-api-error.ts`

This service stores recent `{ id, message }` notifications in a signal and removes an item when the user dismisses it.

### Global UI — `hospitality-client/src/app/core/components/global-http-status.ts`

The component reads loading and error signals and renders the progress bar, spinner pill, and notification stack. It is included in the root `app.html` beside `<router-outlet>`, so it is present on every route.

### Tests — `hospitality-client/src/app/core/interceptors/loading-error.interceptor.spec.ts`

The HTTP testing backend verifies concurrent requests keep the spinner active until the last request completes. It also verifies an error creates a global message, remains the same error for the subscriber, and ends the loading state.

## 7. Run and verify the project

Open a terminal in `hospitality-client` and run:

```powershell
npm install
npm start
npm test -- --watch=false
npm run build
```

Start either `HospitalityHub.Api` (SQL Server) or `HospitalityHub.Api.Sqlite` (SQLite) in another terminal. Confirm the selected `ApiService` base URL, that the backend is running, and that its CORS policy permits the exact Angular origin. A browser CORS error can look like a network error even when the API process is running.

### Interceptor learning checklist

- [ ] I can explain what request middleware does and why an app might use it.
- [ ] I can trace a call from component -> `ApiService` -> `HttpClient` -> interceptor -> backend -> subscriber.
- [ ] I understand why active requests need a counter instead of one boolean.
- [ ] I can explain why `finalize()` clears loading state on success, failure, and cancellation.
- [ ] I can find where `withInterceptors` registers the function.
- [ ] I know that `catchError` must rethrow when the feature still needs to handle the failure.
- [ ] I can run the interceptor tests and explain what each assertion proves.

### Try these exercises

1. Temporarily stop the API and load the Rooms page. Observe the global error notification and the screen's inline error. Start the API again and retry.
2. In the interceptor test, start two requests. Complete one, verify loading remains true, then complete the second and verify loading becomes false.
3. Add a request timeout or retry policy only after deciding how it should interact with the global loading count and error notifications.
4. As a future exercise, add an `HttpContextToken` to suppress a global toast for a deliberate background request, while still rethrowing the error to the caller.

### Common mistakes and how to diagnose them

| Symptom | Check |
| --- | --- |
| No global loading bar or error toast | Ensure the interceptor is passed to `withInterceptors` in `app.config.ts`, and the global status component is rendered by the root app. |
| Loading indicator gets stuck | Ensure every request stream reaches `finalize`; avoid returning a separate Observable that bypasses the original pipeline. |
| Indicator disappears while another call is active | Use an active-request counter, not a shared boolean set to false by each request. |
| Feature no longer handles an HTTP error | Do not return a success-shaped fallback from `catchError`; rethrow the original error. |
| Browser says CORS or status 0 | Check the running API, trusted HTTPS certificate, configured API URL, and exact allowed frontend origin. |
| Duplicate-looking error messages | The global notification is intentional and the page may also show contextual inline feedback. If the UX becomes noisy, design a per-request opt-out with `HttpContextToken`. |

## 8. Level 2 — JWT login: identity and access tokens

A **user account** identifies a person who can use the management app. Registration creates a user record. Login verifies their credentials and, if valid, returns a signed access token.

### Registration and safe password storage

The browser posts an email and password to `POST /api/auth/register`. The API validates the data, normalizes the email for case-insensitive duplicate checks, hashes the password with ASP.NET Core's `PasswordHasher<User>`, and stores only the hash in the `Users` table. It never stores the original password. A unique database index on normalized email is the final protection against duplicate account records.

### Login and JWT token

The browser posts credentials to `POST /api/auth/login`. The API checks the password hash and returns an access token, expiry time, and basic user information. The token is signed with a server-only key. A JWT is encoded and signed, not encrypted: do not put passwords or private data in its claims.

The signing key is intentionally not checked into the repository. Each API project has a .NET user-secrets ID. Configure a strong key locally and run the user-table migration before starting the API:

```powershell
$key = New-Object byte[] 64
[Security.Cryptography.RandomNumberGenerator]::Create().GetBytes($key)
dotnet user-secrets set "Jwt:SigningKey" ([Convert]::ToBase64String($key)) --project HospitalityHub.Api\HospitalityHub.Api.csproj
dotnet ef database update --project HospitalityHub.Api\HospitalityHub.Api.csproj
```

For SQLite, use `HospitalityHub.Api.Sqlite\HospitalityHub.Api.Sqlite.csproj` for both commands. Use a different private key for each API. The API requires at least 32 UTF-8 bytes and validates the signature, issuer, audience, and expiration.

### Protected API and Angular request flow

Room and booking controllers require `[Authorize]`; the auth endpoints allow anonymous access so a user can register and sign in. Angular's route guard sends an unauthenticated browser navigation to `/login`, but a client-side guard is only a UX feature. The API's `[Authorize]` check is the actual security boundary.

After login, `AuthTokenStorage` saves the token and expiry in `localStorage`. `authInterceptor` adds `Authorization: Bearer <token>` only when the request goes to the configured API origin and the token has not expired. It does not leak the token to unrelated origins. A 401 clears the saved token and routes the user to sign in again. Sign out clears both stored values.

Browser `localStorage` is convenient for a learning project but JavaScript running on the same origin can read it. For a production system, consider a Secure, HttpOnly, SameSite cookie and a complete XSS/CSRF threat model. Never store a password in local storage or put the JWT signing key in Angular code.

### Authentication files to explore

- `HospitalityHub.Api/Controllers/AuthController.cs` and `HospitalityHub.Api.Sqlite/Controllers/AuthController.cs` — register and login endpoints.
- `HospitalityHub.Api/Data/Entities/User.cs`, `HospitalityHub.Api.Sqlite/Data/Entities/User.cs`, and each `Data/Migrations/*AddUsers*` — user schema and migrations.
- `HospitalityHub.Api/Program.cs` and `HospitalityHub.Api.Sqlite/Program.cs` — JWT validation middleware and configuration.
- `hospitality-client/src/app/core/services/auth.ts` and `auth-token-storage.ts` — login/register calls and token lifecycle.
- `hospitality-client/src/app/core/interceptors/auth.interceptor.ts` — adds the bearer header.
- `hospitality-client/src/app/core/guards/auth.guard.ts` and `app.routes.ts` — frontend navigation protection.
- `hospitality-client/src/app/features/auth/` — registration and login screens.

## 9. Level 2 — Room search and six-item pagination

The room list searches by room number or type, without case sensitivity. Search runs on the rooms already loaded by the API (client-side filtering). It then slices the matches into pages of six. Page buttons are disabled at the first/last page, and changing the search resets the current page so a user is not left viewing a page number that no longer exists.

This is **client-side pagination**, useful while the full room list is small. For a very large database, send search, page number, and page size to the API and let the database return one page; that avoids downloading every record. The UI still needs the same concepts, but the server should return both the page items and total match count.

### Signals and derived state

The room feature already stores rooms, search text, and current page in Angular signals. `filteredRooms` is a `computed` value derived from rooms and search text. `visibleRooms` is another computed value derived from filtered rooms and current page. The template reads these signals, so updating the search or page automatically updates the displayed cards.

This avoids manually keeping multiple copies of the same state in sync. `rooms` is the source data; filtered/paginated lists and counts are derived values. Booking screens and global loading/error UI also use signals. Signals were introduced in Angular 16 and are part of modern Angular (the project currently uses Angular 21).

### Search/pagination exercises

1. Add 13 rooms to the test data and explain why the pages contain 6, 6, and 1 rooms.
2. Search for `suite`, then try `SUITE`. Explain why both find the same rooms.
3. Go to the last page and delete its only room. Verify the current page is clamped to a valid page.
4. Sketch a future server API such as `GET /api/rooms?search=suite&page=2&pageSize=6` and define a response containing `items`, `page`, `pageSize`, and `totalCount`.

## 10. Checklist and troubleshooting for the new features

- [ ] I can explain the difference between a password, a password hash, and a signed JWT.
- [ ] I know why the JWT signing key belongs only on the server and why user-secrets is used for local development.
- [ ] I can describe registration -> password hashing -> login -> token storage -> bearer request -> API authorization.
- [ ] I understand that a route guard does not replace API authorization.
- [ ] I can explain why the auth interceptor attaches the token only to the configured API origin.
- [ ] I can explain the difference between client-side six-item pagination and server-side pagination.
- [ ] I can identify source signals versus `computed` derived state in the room list.

| Symptom | Check |
| --- | --- |
| API startup says the JWT key is missing | Set `Jwt:SigningKey` with `dotnet user-secrets` in the API project you are running. Do not add it to appsettings or the Angular app. |
| Register/login returns 404 | Ensure the correct .NET API project is running and that it includes `api/auth/register` and `api/auth/login`. |
| Room/booking requests return 401 | Sign in again, check local token expiry, and confirm that the auth interceptor and bearer token are configured. |
| User table does not exist | Run `dotnet ef database update` for the same API project/database you start. |
| Login succeeds but rooms cannot load | Check that both client and API use the same API base URL and that CORS allows the frontend origin. |
| Search results show an empty page | Search resets the page to 1; if adding new logic, clamp current page after result count changes too. |

## 11. Roadmap after this notebook

Level 1 room create/list/delete and booking create/list are substantially complete; booking update/cancellation and room update remain CRUD follow-ups. Level 2 now includes interceptors, JWT registration/login, route guards, search, and pagination. Level 3 signals are already used for room/booking state and computed summaries. Possible next topics are lazy-loaded feature routes, server-side pagination, role-based authorization, and deployment configuration. Choose one topic at a time, add a test, and explain the result in your own words.